## Model Training
Purpose: Make a baseline model full_features.csv that will able to be personalized for each given subject.

### Import Libraries

In [ ]:
import json
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from pathlib import Path 
from datetime import datetime, timezone
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

### Set up directory paths

In [ ]:
PROCCESED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
DOCS_DIR = Path("../docs")
MODELS_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)
(DOCS_DIR / "screenshots").mkdir(exist_ok=True)

LABEL_NAMES = {1: "low", 2: "medium", 3: "high"}


### Load in the features and split off accordingly

In [ ]:

full_features = pd.read_csv(PROCCESED_DIR / "full_features.csv")

non_feature_cols = ["subject", "test", "phase"]
feature_cols = [c for c in full_features.columns if c not in non_feature_cols]

X = full_features[feature_cols]
y = full_features["test"]
groups = full_features["subject"]

print(f"Samples: {len(X)}, Features: {len(feature_cols)}, Subjects: {groups.nunique()}\n")
print(f"Feature-to-sample ratio: {len(feature_cols)/len(X):.3f}")
print("REMINDER: Keep track of this ratio!")

# Double checking that data isn't missing any values
print(f"Infs in ratio: {np.isinf(full_features["frontal_theta_beta_ratio"]).sum()}")
print(f"Nulls in ratio:", full_features["frontal_theta_beta_ratio"].isna().sum())
print(f"Ratio Max: {full_features["frontal_theta_beta_ratio"].max()}")
print("Ratio Min:", full_features["frontal_theta_beta_ratio"].min())

# Listing out variances for viewing and analysis
# Does not matter to the structure of the model or the GUI themselves
variances = X.var()
print("\nLowest feature variances:\n", variances.sort_values().head(10))

#### Within Subject Validation

In [ ]:
subject_metrics = []
all_y_true = []
all_y_pred = []

unique_subjects = groups.unique()
print(f"Running Within-Subject Evaluation for {len(unique_subjects)} subjects...\n")

# 2. Iterate through each person one by one
for subj in unique_subjects:

    # Filter the data for JUST this subject
    subj_mask = groups == subj
    X_subj = X[subj_mask].reset_index(drop=True)
    y_subj = y[subj_mask].reset_index(drop=True)

    # Set up a 5-Fold Split (This is a 80% train / 20% test, done 5 times)
    # Stratified ensures the Low/Medium/High classes are balanced in every test set
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    clf = RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=42
    )

    subj_y_true = []
    subj_y_pred = []

    # 3. Train and Test 5 separate times for this specific person
    for train_idx, test_idx in skf.split(X_subj, y_subj):

        X_train, X_test = X_subj.iloc[train_idx], X_subj.iloc[test_idx]
        y_train, y_test = y_subj.iloc[train_idx], y_subj.iloc[test_idx]

        # Train the personalized model
        clf.fit(X_train, y_train)

        # Test it on their held-out 20%
        preds = clf.predict(X_test)

        subj_y_true.extend(y_test)
        subj_y_pred.extend(preds)

        # Add to the global tracking for a final overarching report
        all_y_true.extend(y_test)
        all_y_pred.extend(preds)

    # 4. Calculate this subject's overall personal accuracy
    subj_acc = accuracy_score(subj_y_true, subj_y_pred)
    subj_f1 = f1_score(subj_y_true, subj_y_pred, average="macro")

    subject_metrics.append({"subject": subj, "accuracy": subj_acc, "macro_f1": subj_f1})

    print(f"{subj} >> Accuracy: {subj_acc:.3f} | Macro F1: {subj_f1:.3f}")

# 5. Summarize the grand results across all personalized models
results_df = pd.DataFrame(subject_metrics)

print("\n" + "=" * 45)
print("FINAL WITHIN-SUBJECT RESULTS")
print("=" * 45)
print(
    f"Average Accuracy:  {results_df['accuracy'].mean():.3f} ± {results_df['accuracy'].std():.3f}"
)
print(
    f"Average Macro F1:  {results_df['macro_f1'].mean():.3f} ± {results_df['macro_f1'].std():.3f}"
)
print("=" * 45)

print("\nOverall Classification Report (Sum of all personal models):")
print(
    classification_report(
        all_y_true, all_y_pred, target_names=["low", "medium", "high"]
    )
)

#### Making an aggregated confusion matrix for my within-subject models

In [ ]:
labels_sorted = sorted(y.unique())
cm = confusion_matrix(all_y_true, all_y_pred, labels=labels_sorted)

fig, ax = plt.subplots(figsize=(6,5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[LABEL_NAMES[c] for c in labels_sorted])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Aggregated Workload Classification (Within Subject Version)")
plt.tight_layout()

plt.savefig(DOCS_DIR / "screenshots" / "confusion_matrix.png", dpi=150)
plt.show()


### Saving within subject metrics

In [ ]:
report = classification_report(all_y_true, all_y_pred, output_dict=True)

metrics = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "n_samples": int(len(X)),
    "n_features": int(len(feature_cols)),
    "n_subjects": int(groups.nunique()),
    "cv_folds": 5, 
    "cv_accuracy": float(results_df['accuracy'].mean()),
    "cv_macro_f1": float(results_df['macro_f1'].mean()),
    "class_low_f1": float(report['1']['f1-score']),
    "class_med_f1": float(report['2']['f1-score']),
    "class_high_f1": float(report['3']['f1-score'])
}

with open(DOCS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
    
print(f"Finalized metrics saved to {DOCS_DIR / "metrics.json"}")